<a href="https://colab.research.google.com/github/nehahemathkar/intro-build-calculator/blob/main/Intelligent_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

In [3]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ").strip()
if not os.environ["GROQ_API_KEY"]:
    raise ValueError("Enter a Groq API key in the hidden prompt.")
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0, timeout=30, max_retries=1)
print("Model client configured. Cell 3 tests the connection.")

Enter your Groq API key: ··········
Model client configured. Cell 3 tests the connection.


In [4]:
APPLES = "One person brings five apples and another brings ten. They put them together. How many apples are there?"
plain_reply = model.invoke([("human", APPLES)])
print(plain_reply.content)

They have **15 apples** in total (5 + 10 = 15).


In [5]:
role_reply = model.invoke([
    ("system", "You are a patient math tutor. Give one short sentence with the answer and its units."),
    ("human", APPLES),
])
print(role_reply.content)

There are 15 apples.


In [6]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.name)

add


In [7]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.name)

tree_distance


In [8]:
print("Apples:", add.invoke({"a": 5, "b": 10}))
print("Tree distance in metres:", tree_distance.invoke({"height_m": 10, "angle_deg": 30}))
print("Invalid angle:", tree_distance.invoke({"height_m": 10, "angle_deg": 0}))

Apples: 15
Tree distance in metres: 17.32
Invalid angle: Invalid angle. Supply an angle strictly between 0 and 90 degrees.


In [9]:
from langchain.agents import create_agent

INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)
print("Agent ready")

Agent ready


In [10]:
from langchain_core.messages import ToolMessage

def solve(problem, *, run_name="intro_calculator"):
    return agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": run_name},
    )

def show_run(problem):
    result = solve(problem)
    call_count = 0
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            call_count += 1
            print("Tool requested:", call["name"], call["args"])
        if isinstance(message, ToolMessage):
            print("Tool returned:", message.name, message.content)
    print("Tool requests:", call_count)
    print("Final answer:", result["messages"][-1].content)
    return result

print("Viewer ready")

Viewer ready


In [11]:
apples_run = show_run(APPLES)

Tool requested: add {'a': 5, 'b': 10}
Tool returned: add 15
Tool requests: 1
Final answer: There are **15 apples**.


In [12]:
TREE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "The angle of elevation to the top is 30 degrees. "
    "How far away is the tree horizontally?"
)
tree_run = show_run(TREE)

Tool requested: tree_distance {'angle_deg': 30, 'height_m': 10}
Tool returned: tree_distance 17.32
Tool requests: 1
Final answer: The horizontal distance from you to the tree is **17.32 m**.


In [13]:
MISSING_ANGLE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "How far away is the tree horizontally?"
)
missing_run = show_run(MISSING_ANGLE)

Tool requests: 0
Final answer: I can calculate the horizontal distance using the tree_distance tool, but I need the angle of elevation (in degrees). Could you please provide that?


In [14]:
from langsmith import Client, tracing_context

destination = input("LangSmith destination type (agent/project): ").strip().lower()
if destination == "agent":
    os.environ.pop("LANGSMITH_PROJECT", None)
    os.environ["LANGSMITH_AGENT_ID"] = input("Agent ID: ").strip()
    os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = input(
        "Environment (for example local): "
    ).strip()
elif destination == "project":
    os.environ.pop("LANGSMITH_AGENT_ID", None)
    os.environ.pop("LANGSMITH_AGENT_ENVIRONMENT", None)
    os.environ["LANGSMITH_PROJECT"] = input("Project name: ").strip()
else:
    raise ValueError("Enter agent or project, then rerun this cell.")

endpoint = input("LangSmith API URL (blank for default US): ").strip()
if endpoint:
    os.environ["LANGSMITH_ENDPOINT"] = endpoint
workspace_id = input("Workspace ID (blank if unnecessary): ").strip()
if workspace_id:
    os.environ["LANGSMITH_WORKSPACE_ID"] = workspace_id
smith_key = getpass("LangSmith API key: ").strip()
smith_client = Client(api_key=smith_key)
print("Tracing client configured.")

LangSmith destination type (agent/project): project
Project name: intro-build-calculator
LangSmith API URL (blank for default US): 
Workspace ID (blank if unnecessary): 
LangSmith API key: ··········
Tracing client configured.


In [15]:
with tracing_context(enabled=True, client=smith_client):
    solve(APPLES, run_name="intro_apples")
    solve(TREE, run_name="intro_tree")
    solve(MISSING_ANGLE, run_name="intro_missing_angle")
smith_client.flush()
print("Look for intro_apples, intro_tree and intro_missing_angle.")

Look for intro_apples, intro_tree and intro_missing_angle.


In [16]:
fresh_run = show_run('One person brings seven apples and another brings eight. How many apples do they have together?')

Tool requested: add {'a': 7, 'b': 8}
Tool returned: add 15
Tool requests: 1
Final answer: They have **15 apples** together.
